# Hafta 2 - Aylık Rapor #1: Ekim 2026

**Soru:** Ekim 2026'da fiyatı hacim mi taşıdı, maliyet mi? Piyasa kışa hangi konumda giriyor?

| Bölüm | Alt soru | Yöntem |
|---|---|---|
| 1 | **Konum:** Ekim 2026, geçmiş Ekim'lere ve tüm aylara göre nerede? | Ay kartı: Ekim'ler arasında sıra, 46 ay içinde yüzdelik |
| 2 | **Ayrıştırma:** Değişim hacimden mi, maliyetten mi? | Termik yük - fiyat eğrisi; eğri boyunca hareket (hacim) ve eğrinin kayması (maliyet) |
| 3 | **Maliyet kontrolü** (opsiyonel): Eğri kayması gaz maliyetiyle uyumlu mu? | BOTAŞ elektrik üretimi tarifesi, USD'ye çevrilmiş |
| 4 | **Dengeleme:** Gün öncesi tahminler ne kadar tuttu? | SMF-PTF farkı, sistem yönü, GİP-PTF farkı |
| 5 | **Kasım'da şu ana kadar** | Hafta içi günler, aynı günlerin geçmiş yılları |

**İlkeler (Hafta 1'den):** Fiyatlar USD bazında ve aynı ay karşılaştırmasıyla. Modelde olmayan değişken hakkında nicel iddia yok. Sonuçlar ilişkidir.

**Girdi:** `02_islenmis/ana_tablo_h02_saatlik.csv` (01_hafta02_veri_cekme.ipynb, `DENEME_MODU=False` ile)

In [ ]:
# 0) Ayarlar, stil ve veri
import json, warnings, logging
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore"); logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

HEDEF_YIL, HEDEF_AY = 2026, 10           # raporlanan ay
AY_ADI = "Ekim"
CSV = Path("02_islenmis/ana_tablo_h02_saatlik.csv")
OUT = Path("03_grafikler/hafta02"); OUT.mkdir(parents=True, exist_ok=True)

NV, OR, G1, G2, G3 = "#0a1f6b", "#e8590c", "#c3d0ee", "#7f9bd6", "#2f4fb0"
plt.rcParams.update({"font.family": "DejaVu Sans Mono", "font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": NV, "axes.linewidth": 1.5, "axes.labelcolor": NV, "xtick.color": NV, "ytick.color": NV,
    "axes.grid": True, "grid.color": "#c9d4ee", "grid.linestyle": ":", "axes.axisbelow": True, "figure.dpi": 100})
YIL_RENK = {2023: G1, 2024: G2, 2025: G3, 2026: OR}
def kaydet(fig, ad): fig.savefig(OUT / f"{ad}.png", dpi=200, facecolor="white", bbox_inches="tight"); plt.show()

df = pd.read_csv(CSV, parse_dates=["zaman"]).set_index("zaman").sort_index()
# Geçmiş yıllar tabloda yoksa (ör. veri DENEME_MODU ile çekildiyse) PTF/üretim/tüketim geçmişi Hafta 1 tablosundan tamamlanır
H01 = Path("../Hafta_01/02_islenmis/ana_tablo_saatlik.csv")
if df.index.min() > pd.Timestamp("2023-01-31") and H01.exists():
    h1 = pd.read_csv(H01, parse_dates=["zaman"]).set_index("zaman")
    df = pd.concat([h1[h1.index < df.index.min()], df]).sort_index()
    print("Bilgi: PTF/üretim/tüketim geçmişi Hafta 1 tablosundan eklendi. SMF/sistem yönü/GİP geçmişi için veri çekmeyi DENEME_MODU=False ile çalıştırın.")
if (df.index.min() > pd.Timestamp(f"{HEDEF_YIL-1}-{HEDEF_AY:02d}-01")):
    raise SystemExit("Geçmiş yılların verisi yok. 01_hafta02_veri_cekme.ipynb'yi DENEME_MODU = False ile çalıştırın (Hafta_01 klasörü de yanında olmalı).")
df["y"], df["m"], df["h"] = df.index.year, df.index.month, df.index.hour
U = lambda c: df[c] if c in df else 0.0
df["hidro"]  = U("uretim_dammedHydro") + U("uretim_river")
df["gaz"]    = U("uretim_naturalGas") + U("uretim_lng")
df["kur"]    = (df.ptf / df.ptf_usd).where(df.ptf_usd > 1)          # EPİAŞ'ın kullandığı TL/USD (örtük)
df["kur"]    = df.kur.groupby(df.index.date).transform("median")     # gün içi tek değer
# Termik yük: tüketimin hidro, rüzgâr, güneş, jeotermal ve biyokütle dışında kalan kısmı (yaklaşık)
df["termik_yuk"] = df.tuketim_mwh - df.hidro - U("uretim_wind") - U("uretim_solar") - U("uretim_geothermal") - U("uretim_biomass")
for c in ["smf", "gip"]:
    if c in df: df[c + "_usd"] = df[c] / df.kur
VAR = {c: c in df and df.loc[f"{HEDEF_YIL-1}-{HEDEF_AY:02d}", c].notna().mean() > 0.5 if c in df else False for c in ["smf", "gip", "sistem_yonu"]}
if not all(VAR.values()): print("Uyarı: geçmiş yıllar için eksik veri:", [k for k, v in VAR.items() if not v], "-> bu kısımlar atlanır.")
SON_TAM_AY = (df.index.max() + pd.Timedelta(hours=1)).to_period("M") - 1
print("Veri:", df.index.min(), "-", df.index.max(), "| saat:", len(df), "| son tam ay:", SON_TAM_AY, "| yeni veri:", VAR)

## 1) Konum: ay kartı
Her gösterge için: (a) Ekim 2026'nın 2023-2026 Ekim'leri arasındaki sırası, (b) tüm tam aylar içindeki yüzdelik konumu.
Fiyatlar USD bazında. Paylar, toplam üretim içindeki saatlik payların ay ortalamasıdır.

In [ ]:
tam = df[df.index.to_period("M") <= SON_TAM_AY]
g = tam.groupby(["y", "m"])
aylik = pd.DataFrame({
    "PTF (USD/MWh)":             g.ptf_usd.mean(),
    "Akşam/öğle fiyat oranı":     tam[tam.h.between(17, 21)].groupby(["y", "m"]).ptf_usd.mean()
                                  / tam[tam.h.between(10, 15)].groupby(["y", "m"]).ptf_usd.mean(),
    "Tüketim (GW)":              g.tuketim_mwh.mean() / 1000,
    "Termik yük (GW)":           g.termik_yuk.mean() / 1000,
    "Hidro payı (%)":            (tam.hidro / tam.uretim_total).groupby([tam.y, tam.m]).mean() * 100,
    "Rüzgâr payı (%)":           (tam.uretim_wind / tam.uretim_total).groupby([tam.y, tam.m]).mean() * 100,
    "Güneş payı (%)*":           (tam.uretim_solar / tam.uretim_total).groupby([tam.y, tam.m]).mean() * 100,
    "Doğal gaz payı (%)":        (tam.gaz / tam.uretim_total).groupby([tam.y, tam.m]).mean() * 100,
    "İthal kömür payı (%)":      (tam.uretim_importCoal / tam.uretim_total).groupby([tam.y, tam.m]).mean() * 100,
    "TL/USD":                    g.kur.mean(),
})
if VAR["smf"]: aylik["SMF − PTF (USD/MWh)"] = g.smf_usd.mean() - g.ptf_usd.mean()
if VAR["sistem_yonu"]: aylik["Enerji açığı saatleri (%)"] = g.sistem_yonu.apply(lambda s: (s == "acik").mean() * 100)
if VAR["gip"]: aylik["GİP − PTF (USD/MWh)"] = g.gip_usd.mean() - g.ptf_usd.mean()

hedef = (HEDEF_YIL, HEDEF_AY)
ayni_ay = aylik.xs(HEDEF_AY, level="m")
kart = pd.DataFrame({
    f"{AY_ADI} {HEDEF_YIL}": aylik.loc[hedef],
    f"{AY_ADI} {HEDEF_YIL-1}": aylik.loc[(HEDEF_YIL - 1, HEDEF_AY)],
    f"Geçmiş {AY_ADI} ort.": ayni_ay.drop(HEDEF_YIL).mean(),
    f"{AY_ADI}'ler içinde sıra (büyükten)": ayni_ay.rank(ascending=False).loc[HEDEF_YIL].astype(int).astype(str) + f"/{len(ayni_ay)}",
    "Tüm aylar içinde yüzdelik": (aylik.rank(pct=True).loc[hedef] * 100).round(0).astype(int),
})
pd.set_option("display.width", 200); display(kart.round(2))
print("* Güneş: EPİAŞ gerçek zamanlı üretimdeki 'sun' alanı; lisanssız üretimi tam kapsamayabilir.")

In [ ]:
# Ay kartı grafiği: her satırda tüm aylar (gri), geçmiş aynı aylar (mavi), hedef ay (turuncu)
satirlar = list(aylik.columns)
fig, ax = plt.subplots(figsize=(11, 0.62 * len(satirlar) + 1.2))
for i, c in enumerate(satirlar):
    s = aylik[c].dropna(); sr = np.sort(s.values)
    n = lambda v: (np.searchsorted(sr, v, side="right") - .5) / len(sr)   # konum = yüzdelik sıra (uç değerler ölçeği bozmaz)
    y = len(satirlar) - 1 - i
    ax.hlines(y, 0, 1, color=G1, lw=1)
    ax.scatter([n(v) for v in s.values], [y] * len(s), s=14, color="#9aa1ad", alpha=.55, zorder=2)
    for yy, v in ayni_ay[c].drop(HEDEF_YIL, errors="ignore").items():
        ax.scatter(n(v), y, s=60, color=YIL_RENK.get(yy, G2), edgecolor=NV, lw=.6, zorder=3)
        ax.text(n(v), y + .22, str(yy)[2:], ha="center", fontsize=7.5, color=NV)
    v = aylik.loc[hedef, c]
    ax.scatter(n(v), y, s=150, color=OR, edgecolor="black", lw=1, zorder=4, marker="D")
    ax.text(-0.02, y, c, ha="right", va="center", fontsize=10, color=NV)
    ax.text(1.03, y, f"{v:,.1f}".replace(",", "X").replace(".", ",").replace("X", "."), ha="left", va="center", fontsize=10.5, color=OR, fontweight="bold")
    ax.text(1.16, y, f"%{kart.loc[c, 'Tüm aylar içinde yüzdelik']}", ha="left", va="center", fontsize=9.5, color=NV)
ax.set_xlim(-0.02, 1.25); ax.set_ylim(-.9, len(satirlar) - .3); ax.axis("off")
ax.text(0.5, -0.65, "Yatay konum: tüm aylar içindeki yüzdelik sıra. Gri: tüm aylar, mavi: geçmiş aynı aylar, turuncu: hedef ay.", fontsize=8.5, color="#666", ha="center")
ax.text(0, len(satirlar) - .1, "düşük", fontsize=8.5, color="#666"); ax.text(1, len(satirlar) - .1, "yüksek", fontsize=8.5, color="#666", ha="right")
ax.text(1.03, len(satirlar) - .1, "değer", fontsize=8.5, color="#666"); ax.text(1.16, len(satirlar) - .1, "yüzdelik", fontsize=8.5, color="#666")
kaydet(fig, "h02_ay_karti")

## 2) Ayrıştırma: hacim mi, maliyet mi?
**Fikir (merit order):** Termik santrallerin karşılaması gereken yük (termik yük) arttıkça teklif sırasında daha pahalı santrallere çıkılır. Her ay için "termik yük → PTF" eğrisi çizilir.
- **Hacim etkisi:** Yük değişti, aynı eğri üzerinde ilerledik.
- **Eğri kayması:** Aynı yükte fiyat değişti. Maliyet (gaz, kömür, kur), teklif davranışı ve ithalat/ihracat gibi etkenleri birlikte içerir; **tek başına "maliyet" demek için 3. bölümle desteklenmelidir.**

Her ay için eğri, 1 GW'lık yük dilimlerindeki medyan fiyattan kurulur ve monoton artan olacak şekilde düzeltilir (merit order mantığı). Ayrıştırma iki sırayla hesaplanıp ortalaması alınır (sıra etkisini nötrlemek için). Eğri dışına taşan (ekstrapolasyon) saatlerin oranı raporlanır.

**Not:** Eğrilerin üst kısmındaki düz bölümler azami fiyat limitinden (tavan) gelir. Tavana takılan saatlerde fiyat yükle artamaz; bu, yüksek yükte hacim etkisini olduğundan küçük gösterebilir.

In [ ]:
def ay_veri(y, m):
    s = df[(df.y == y) & (df.m == m)].dropna(subset=["termik_yuk", "ptf_usd"])
    return s.termik_yuk.values / 1000, s.ptf_usd.values

def egri(x, p, bin_gw=1.0, min_n=10):
    """Monoton artan arz eğrisi: 1 GW'lık dilimlerde medyan fiyat, ardından kümülatif maksimum (merit order artan olmalı).
    Eğrinin dışındaki yüklerde uç değer sabit tutulur."""
    b = np.floor(x / bin_gw) * bin_gw + bin_gw / 2
    t = pd.DataFrame({"b": b, "p": p}).groupby("b").p.agg(["median", "size"])
    t = t[t["size"] >= min_n]
    xs, ys = t.index.values, np.maximum.accumulate(t["median"].values)
    return (lambda z: np.interp(z, xs, ys)), (xs.min(), xs.max())

def ayristir(a, b):
    """a: (yıl, ay) başlangıç, b: (yıl, ay) bitiş. USD/MWh cinsinden."""
    xa, pa = ay_veri(*a); xb, pb = ay_veri(*b)
    fa, ra = egri(xa, pa); fb, rb = egri(xb, pb)
    xb_c, xa_c = np.clip(xb, *ra), np.clip(xa, *rb)
    h1 = fa(xb_c).mean() - fa(xa).mean();  k1 = fb(xb).mean() - fa(xb_c).mean()   # sıra 1: önce hacim
    k2 = fb(xa_c).mean() - fa(xa).mean();  h2 = fb(xb).mean() - fb(xa_c).mean()   # sıra 2: önce kayma
    return dict(baslangic=pa.mean(), bitis=pb.mean(), fark=pb.mean() - pa.mean(),
                hacim=(h1 + h2) / 2, kayma=(k1 + k2) / 2, hacim_aralik=(min(h1, h2), max(h1, h2)),
                yuk_degisim_gw=xb.mean() - xa.mean(),
                ekstrapolasyon=float(((xb < ra[0]) | (xb > ra[1])).mean() + ((xa < rb[0]) | (xa > rb[1])).mean()) / 2)

KARS = {f"{AY_ADI} {HEDEF_YIL-1} → {AY_ADI} {HEDEF_YIL}": ((HEDEF_YIL - 1, HEDEF_AY), hedef),
        f"Eylül {HEDEF_YIL} → {AY_ADI} {HEDEF_YIL}": ((HEDEF_YIL, HEDEF_AY - 1), hedef)}
ayr = pd.DataFrame({k: ayristir(*v) for k, v in KARS.items()}).T
display(ayr[["baslangic", "bitis", "fark", "hacim", "kayma", "yuk_degisim_gw", "ekstrapolasyon"]].astype(float).round(2))
print("hacim_aralik: iki sıralamanın verdiği hacim etkisi aralığı (dar olması iyi):"); print(ayr.hacim_aralik.to_string())

In [ ]:
# Grafik A: termik yük - fiyat eğrileri (aynı ay, yıllar)
fig, ax = plt.subplots(figsize=(10, 6))
for yy in sorted(df.y.unique()):
    if (yy, HEDEF_AY) not in aylik.index: continue
    x, p = ay_veri(yy, HEDEF_AY); f, r = egri(x, p); z = np.linspace(*r, 50)
    ax.scatter(x, p, s=5, color=YIL_RENK.get(yy, G2), alpha=.2)
    ax.plot(z, f(z), color=YIL_RENK.get(yy, G2), lw=3.4 if yy == HEDEF_YIL else 2.2, label=f"{AY_ADI} {yy}")
ax.set_xlabel("Termik yük (GW) = tüketim - hidro - rüzgâr - güneş - jeotermal - biyokütle")
ax.set_ylabel("Saatlik PTF (USD/MWh)"); ax.legend(frameon=False, loc="upper left"); ax.set_ylim(bottom=0)
kaydet(fig, "h02_arz_egrisi")

# Grafik B: ayrıştırma (şelale)
fig, axs = plt.subplots(1, len(ayr), figsize=(11, 5), sharey=True)
for ax, (ad, r) in zip(np.atleast_1d(axs), ayr.iterrows()):
    adim = [("Başlangıç", r.baslangic, 0, G2), ("Hacim", r.hacim, r.baslangic, NV), ("Eğri kayması", r.kayma, r.baslangic + r.hacim, OR), ("Bitiş", r.bitis, 0, G3)]
    for i, (et, v, alt, c) in enumerate(adim):
        ax.bar(i, v, bottom=alt, color=c, width=.6)
        ax.text(i, alt + v + (1.5 if v >= 0 else -4), f"{v:+.1f}" if et in ("Hacim", "Eğri kayması") else f"{v:.1f}", ha="center", fontsize=10)
    ax.set_xticks(range(4)); ax.set_xticklabels([a[0] for a in adim], fontsize=9); ax.set_title(ad, fontsize=11, loc="left", color=NV, pad=14)
uc = [v for _, r in ayr.iterrows() for v in (r.baslangic, r.bitis, r.baslangic + r.hacim)]
np.atleast_1d(axs)[0].set_ylim(max(0, min(uc) - 15), max(uc) + 10)
np.atleast_1d(axs)[0].set_ylabel("Ortalama PTF (USD/MWh), eksen sıfırdan başlamaz")
kaydet(fig, "h02_ayristirma")

## 3) Maliyet kontrolü (opsiyonel): BOTAŞ elektrik üretimi tarifesi
Eğri kaymasını "maliyet" diye okuyabilmek için gaz maliyetinin aynı yönde hareket ettiğini görmek gerekir.

**Yapmanız gereken:** BOTAŞ'ın yayımladığı aylık tarife tablolarından **elektrik üretim amaçlı** doğal gaz fiyatını (TL / 1000 Sm³) alıp `botas_tarife.csv` dosyasına yazın. Hücre şablonu oluşturur; doldurulmazsa bu bölüm atlanır. Değerleri BOTAŞ'ın resmi tarife sayfasından doğrulayın.

**Yaklaşık dönüşüm:** Kombine çevrim santrali 1 MWh elektrik için yaklaşık **175 Sm³** gaz yakar (~%54 verim, 1 Sm³ ≈ 10,64 kWh). Bu bir varsayımdır; santral verimine göre ±%15 değişir. Karbon maliyeti ve diğer giderler dahil değildir.

In [ ]:
TARIFE = Path("botas_tarife.csv"); SM3_PER_MWH = 175
if not TARIFE.exists():
    aylar = pd.period_range("2023-01", f"{HEDEF_YIL}-{HEDEF_AY:02d}", freq="M").astype(str)
    pd.DataFrame({"ay": aylar, "tarife_tl_1000sm3": ""}).to_csv(TARIFE, index=False)
    print("Şablon oluşturuldu:", TARIFE, "-> doldurup bu hücreyi tekrar çalıştırın.")
t = pd.read_csv(TARIFE)
t["tarife_tl_1000sm3"] = pd.to_numeric(t.tarife_tl_1000sm3.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")
t = t.dropna()
MALIYET_VAR = len(t) >= 6
if MALIYET_VAR:
    t["y"] = t.ay.str[:4].astype(int); t["m"] = t.ay.str[5:7].astype(int); t = t.set_index(["y", "m"])
    kur_ay = aylik["TL/USD"]
    t["gaz_maliyeti_usd_mwh"] = t.tarife_tl_1000sm3 / 1000 * SM3_PER_MWH / kur_ay.reindex(t.index)
    gaz_saat = tam[tam.gaz / tam.uretim_total > 0.25].groupby(["y", "m"]).ptf_usd.mean()   # gazın marjinal olma ihtimali yüksek saatler
    m = pd.DataFrame({"PTF (gazın >%25 olduğu saatler)": gaz_saat, "Gaz yakıt maliyeti (yaklaşık)": t.gaz_maliyeti_usd_mwh}).dropna()
    m["Fark (kaba marj)"] = m.iloc[:, 0] - m.iloc[:, 1]
    display(m.xs(HEDEF_AY, level="m").round(1))
    for ad, (a, b) in KARS.items():
        if a in t.index and b in t.index:
            print(f"{ad}: gaz maliyeti değişimi {t.gaz_maliyeti_usd_mwh[b] - t.gaz_maliyeti_usd_mwh[a]:+.1f} USD/MWh | eğri kayması {ayr.loc[ad, 'kayma']:+.1f} USD/MWh")
    fig, ax = plt.subplots(figsize=(10, 5))
    xs = [f"{y}-{mm:02d}" for y, mm in m.index]
    ax.plot(xs, m.iloc[:, 0], color=NV, lw=2.2, label=m.columns[0]); ax.plot(xs, m.iloc[:, 1], color=OR, lw=2.6, label=m.columns[1])
    ax.set_xticks(xs[::3]); ax.tick_params(axis="x", rotation=45); ax.set_ylabel("USD/MWh"); ax.legend(frameon=False)
    kaydet(fig, "h02_gaz_maliyeti")
else:
    print("Tarife verisi yok (en az 6 ay gerekli): bu bölüm atlandı. Eğri kayması 'maliyet' diye yorumlanmamalı.")

## 4) Dengeleme: gün öncesi ne kadar tuttu?
- **SMF − PTF:** Pozitifse dengeleme gerçek zamanda gün öncesinden pahalı, yani sistem çoğunlukla açık vermiş.
- **Sistem yönü:** Enerji açığı / fazlası / dengede saatlerin oranı.
- **GİP − PTF:** Gün içinde pozisyon düzeltmenin gün öncesine göre primi.

In [ ]:
if VAR["smf"] or VAR["sistem_yonu"]:
    s = df[df.m == HEDEF_AY]
    tablo = pd.DataFrame(index=sorted(s.y.unique()))
    if VAR["smf"]:
        tablo["SMF − PTF (USD)"] = s.groupby("y").apply(lambda d: (d.smf_usd - d.ptf_usd).mean())
        tablo["|SMF − PTF| (USD)"] = s.groupby("y").apply(lambda d: (d.smf_usd - d.ptf_usd).abs().mean())
    if VAR["sistem_yonu"]:
        yon = s.groupby("y").sistem_yonu.value_counts(normalize=True).unstack().fillna(0) * 100
        tablo = tablo.join(yon.rename(columns={"acik": "Açık (%)", "fazla": "Fazla (%)", "denge": "Denge (%)"}))
    if VAR["gip"]: tablo["GİP − PTF (USD)"] = s.groupby("y").apply(lambda d: (d.gip_usd - d.ptf_usd).mean())
    tablo.index = [f"{AY_ADI} {y}" for y in tablo.index]; display(tablo.round(1))

    fig, axs = plt.subplots(1, 2, figsize=(11, 4.8))
    if VAR["sistem_yonu"]:
        alt = np.zeros(len(yon))
        for k, c, et in [("acik", OR, "Açık"), ("denge", G1, "Denge"), ("fazla", NV, "Fazla")]:
            if k in yon: axs[0].bar([str(y) for y in yon.index], yon[k], bottom=alt, color=c, label=et); alt += yon[k].values
        axs[0].set_ylabel(f"{AY_ADI} saatlerinin dağılımı (%)"); axs[0].legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(.5, 1.12))
        h = s[s.y == HEDEF_YIL].groupby("h").sistem_yonu.apply(lambda x: (x == "acik").mean() * 100)
        axs[1].bar(h.index, h.values, color=OR); axs[1].set_xlabel("Saat"); axs[1].set_ylabel(f"Enerji açığı saat oranı (%), {AY_ADI} {HEDEF_YIL}")
    kaydet(fig, "h02_dengeleme")
else:
    print("SMF / sistem yönü verisi yok.")

## 5) Kasım'da şu ana kadar
Sadece **hafta içi** günler karşılaştırılır (hafta sonu ve 29 Ekim / 28 Ekim yarım gün dışarıda). Kasım 2026'nın mevcut günleri, Ekim 2026 hafta içi ortalaması ve önceki yılların **aynı Kasım günleriyle** kıyaslanır. Bu bir tahmin değildir.

In [ ]:
kasim = df[(df.y == HEDEF_YIL) & (df.m == HEDEF_AY + 1)]
if len(kasim) >= 72:
    son_gun = kasim.index.max().day
    haftaici = lambda d: d[(d.index.dayofweek < 5) & ~((d.m == 10) & d.index.day.isin([28, 29]))]
    ref_ekim = haftaici(df[(df.y == HEDEF_YIL) & (df.m == HEDEF_AY)])
    kasim_hi = haftaici(kasim)
    gecmis = haftaici(df[(df.m == HEDEF_AY + 1) & (df.y < HEDEF_YIL) & (df.index.day <= son_gun)])
    ozet_k = pd.DataFrame({
        f"{AY_ADI} {HEDEF_YIL} (hafta içi)": [ref_ekim.ptf_usd.mean(), ref_ekim.ptf.mean(), ref_ekim.tuketim_mwh.mean()/1000, (ref_ekim.hidro/ref_ekim.uretim_total).mean()*100, (ref_ekim.gaz/ref_ekim.uretim_total).mean()*100],
        f"Kasım 1-{son_gun} {HEDEF_YIL} (hafta içi)": [kasim_hi.ptf_usd.mean(), kasim_hi.ptf.mean(), kasim_hi.tuketim_mwh.mean()/1000, (kasim_hi.hidro/kasim_hi.uretim_total).mean()*100, (kasim_hi.gaz/kasim_hi.uretim_total).mean()*100],
        f"Kasım 1-{son_gun}, {gecmis.y.min()}-{gecmis.y.max()} ort.": [gecmis.ptf_usd.mean(), np.nan, gecmis.tuketim_mwh.mean()/1000, (gecmis.hidro/gecmis.uretim_total).mean()*100, (gecmis.gaz/gecmis.uretim_total).mean()*100],
    }, index=["PTF (USD/MWh)", "PTF (TL/MWh)", "Tüketim (GW)", "Hidro payı (%)", "Doğal gaz payı (%)"])
    ozet_k["Kasım/Ekim değişim (%)"] = (ozet_k.iloc[:, 1] / ozet_k.iloc[:, 0] - 1) * 100
    display(ozet_k.round(1))
    print("Not: TL değişimi kurdaki hareketi de içerir; yorumu USD satırıyla yapın.")
    gun = df[(df.y == HEDEF_YIL) & (df.m >= HEDEF_AY)].ptf_usd.resample("D").mean()
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(gun.index, gun.values, color=OR, lw=2.4, marker="o", ms=3)
    ax.axhline(ref_ekim.ptf_usd.mean(), color=NV, ls="--", lw=1.2, label=f"{AY_ADI} {HEDEF_YIL} hafta içi ort.")
    ax.axvline(pd.Timestamp(HEDEF_YIL, HEDEF_AY + 1, 1), color="#999", lw=1)
    ax.set_ylabel("Günlük ortalama PTF (USD/MWh)"); ax.legend(frameon=False); fig.autofmt_xdate()
    kaydet(fig, "h02_kasim_ilk_gunler")
else:
    ozet_k = None; print("Kasım verisi yok ya da 3 günden az.")

## 6) Eğri neden kaydı? Olası nedenlerin testi
Aynı termik yükte fiyatın değişmesi (eğri kayması) için olası nedenler tek tek sınanır:

| Neden | Test | Kayma bu nedenden geliyorsa beklenen |
|---|---|---|
| Yakıt tarifesi | BOTAŞ elektrik üretim tarifesi, USD karşılığı | Gaz maliyeti aynı yönde değişmeli |
| Kur | Ayrıştırmayı TL ile tekrarla | Kayma TL'de kaybolmalı |
| Fiyat tavanı | Tavana takılan saat oranı; bu saatler hariç ayrıştırma | Tavan saatleri çıkınca kayma küçülmeli |
| Dış ticaret | Net ithalat/ihracat; yük olarak gerçekleşen termik üretim | İhracat artmış olmalı; yeni yükte kayma küçülmeli |
| Termik sıra / bulunabilirlik | Aynı termik üretim düzeyinde gaz ve ithal kömür payı | Aynı yükte daha pahalı (gaz) santral payı artmalı |
| Gün içi dağılım | Ayrıştırmayı gece / gündüz / akşam ayrı yap | Kaymanın hangi saatlerde yoğunlaştığı |

Test edilemeyenler: spot gaz (GRF) ve ithal kömür fiyatı, arıza-bakım kayıtları, baraj doluluk (su değeri). Bunlar ayrıca veri gerektirir.

In [ ]:
TER = [c for c in ["uretim_naturalGas","uretim_lng","uretim_importCoal","uretim_lignite","uretim_blackCoal","uretim_asphaltiteCoal","uretim_fueloil","uretim_naphta"] if c in df]
df["termik_uretim"] = df[TER].sum(axis=1)

def ayristir2(a, b, yuk="termik_yuk", fiyat="ptf_usd", filt=None):
    def get(y, m):
        s = df[(df.y == y) & (df.m == m)].dropna(subset=[yuk, fiyat])
        if filt is not None: s = filt(s)
        return s[yuk].values / 1000, s[fiyat].values
    xa, pa = get(*a); xb, pb = get(*b); fa, ra = egri(xa, pa); fb, rb = egri(xb, pb)
    xbc, xac = np.clip(xb, *ra), np.clip(xa, *rb)
    h = ((fa(xbc).mean() - fa(xa).mean()) + (fb(xb).mean() - fb(xac).mean())) / 2
    k = ((fb(xb).mean() - fa(xbc).mean()) + (fb(xac).mean() - fa(xa).mean())) / 2
    return pd.Series({"fark": pb.mean() - pa.mean(), "hacim": h, "kayma": k})

tavan_disi = lambda s: s[s.ptf < s.ptf.max() - 0.01]
testler = {}
for ad, (a, b) in KARS.items():
    ka, kb = [df[(df.y == y) & (df.m == m)] for y, m in (a, b)]
    r = {"Kayma (USD/MWh)": ayristir2(a, b).kayma,
         "Kayma, TL ile (TL/MWh)": ayristir2(a, b, fiyat="ptf").kayma,
         "Kur değişimi (%)": (kb.kur.median() / ka.kur.median() - 1) * 100,
         "Tavan saatleri (%) başlangıç → bitiş": f"{(ka.ptf >= ka.ptf.max()-0.01).mean()*100:.1f} → {(kb.ptf >= kb.ptf.max()-0.01).mean()*100:.1f}",
         "Tavan USD (başlangıç → bitiş)": f"{ka.ptf.max()/ka.kur.median():.1f} → {kb.ptf.max()/kb.kur.median():.1f}",
         "Kayma, tavan saatleri hariç": ayristir2(a, b, filt=tavan_disi).kayma,
         "Net ithalat (GW) başlangıç → bitiş": f"{ka.uretim_importExport.mean()/1000:.2f} → {kb.uretim_importExport.mean()/1000:.2f}" if "uretim_importExport" in df else "-",
         "Kayma, yük = gerçekleşen termik üretim": ayristir2(a, b, yuk="termik_uretim").kayma}
    for et, f_ in [("gece 22-05", lambda s: s[~s.h.between(6, 21)]), ("gündüz 06-16", lambda s: s[s.h.between(6, 16)]), ("akşam 17-21", lambda s: s[s.h.between(17, 21)])]:
        r[f"Kayma, {et}"] = ayristir2(a, b, filt=f_).kayma
    if MALIYET_VAR and a in t.index and b in t.index:
        r["Gaz yakıt maliyeti değişimi (USD/MWh)"] = t.gaz_maliyeti_usd_mwh[b] - t.gaz_maliyeti_usd_mwh[a]
    testler[ad] = r
testler = pd.DataFrame(testler); display(testler)
print("Not: net ithalat pozitifse ithalat, negatifse ihracat (EPİAŞ importExport alanı).")

In [ ]:
# Termik sıra: aynı termik üretim düzeyinde hangi yakıt üretiyor?
kut = [0, 14, 16, 18, 20, 22, 40]; et = ["<14", "14-16", "16-18", "18-20", "20-22", ">22"]
karisim = {}
for y, m in sorted({p for v in KARS.values() for p in v}):
    s = df[(df.y == y) & (df.m == m)].copy(); s["b"] = pd.cut(s.termik_uretim / 1000, kut, labels=et)
    karisim[(y, m)] = s.groupby("b", observed=False).apply(lambda x: pd.Series({
        "gaz %": (x.gaz / x.termik_uretim).mean() * 100, "ithal kömür %": (x.uretim_importCoal / x.termik_uretim).mean() * 100,
        "PTF (USD)": x.ptf_usd.mean(), "saat": len(x)}))
display(pd.concat(karisim, axis=1).round(1))
ikomur_p95 = {f"{y}-{m:02d}": df[(df.y == y) & (df.m == m)].uretim_importCoal.quantile(.95) / 1000 for y, m in sorted({p for v in KARS.values() for p in v})}
print("İthal kömür üretiminin üst seviyesi (95. yüzdelik, GW):", {k: round(v, 2) for k, v in ikomur_p95.items()})

a, b = KARS[f"Eylül {HEDEF_YIL} → {AY_ADI} {HEDEF_YIL}"]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.8))
x = np.arange(len(et)); w = .38
axs[0].bar(x - w/2, karisim[a]["gaz %"], w, color=G2, label=f"Eylül {a[0]}"); axs[0].bar(x + w/2, karisim[b]["gaz %"], w, color=OR, label=f"{AY_ADI} {b[0]}")
axs[0].set_xticks(x); axs[0].set_xticklabels(et); axs[0].set_xlabel("Termik üretim (GW)"); axs[0].set_ylabel("Termik üretimde doğal gaz payı (%)"); axs[0].legend(frameon=False)
for (yy, mm), c in [(a, G2), (b, OR), ((HEDEF_YIL - 1, HEDEF_AY), G1)]:
    s = df[(df.y == yy) & (df.m == mm)]; cap = s.ptf.max()
    axs[1].plot(range(24), s.groupby("h").ptf.apply(lambda p: (p >= cap - .01).mean() * 100), color=c, lw=2.6, label=f"{yy}-{mm:02d}")
axs[1].set_xlabel("Saat"); axs[1].set_ylabel("Tavan fiyata takılan saat oranı (%)"); axs[1].legend(frameon=False)
kaydet(fig, "h02_nedenler")

## 7) Bulgular (otomatik metin) ve özet dosyası
Aşağıdaki cümleler hesaplanan sayılardan üretilir. Yayınlamadan önce her birini grafiklerle karşılaştırın.

In [ ]:
def f(v, d=1): return f"{v:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
k = kart; c0, c1, c2 = k.columns[:3]
B = []
B.append(f"{AY_ADI} {HEDEF_YIL} ortalama PTF {f(k.loc['PTF (USD/MWh)', c0])} USD/MWh; {AY_ADI} {HEDEF_YIL-1}: {f(k.loc['PTF (USD/MWh)', c1])}, geçmiş {AY_ADI} ort.: {f(k.loc['PTF (USD/MWh)', c2])}. "
         f"{AY_ADI}'ler içinde sıra (büyükten): {k.loc['PTF (USD/MWh)', k.columns[3]]}.")
for g_ in ["Tüketim (GW)", "Termik yük (GW)", "Hidro payı (%)", "Doğal gaz payı (%)", "Rüzgâr payı (%)"]:
    B.append(f"{g_}: {f(k.loc[g_, c0])} ({AY_ADI} {HEDEF_YIL-1}: {f(k.loc[g_, c1])}).")
for ad, r in ayr.iterrows():
    B.append(f"{ad}: PTF {f(r.fark, 1)} USD/MWh değişti. Termik yükteki değişim ({f(r.yuk_degisim_gw, 2)} GW) eğri boyunca {f(r.hacim, 1)} USD/MWh'lık, "
             f"eğrinin kayması {f(r.kayma, 1)} USD/MWh'lık değişimle uyumlu. Ekstrapolasyon oranı %{f(r.ekstrapolasyon*100, 0)}.")
    lo, hi = r.hacim_aralik
    if (hi - lo) > max(abs(r.fark) / 2, 2) or r.ekstrapolasyon > 0.15:
        B.append(f"  DİKKAT ({ad}): İki sıralama farklı sonuç veriyor (hacim etkisi {f(lo)} ile {f(hi)} arası) ya da ekstrapolasyon yüksek. "
                 "Bu ayrıştırmayı nicel olarak yayınlamayın; sadece yön olarak kullanın.")
if not MALIYET_VAR: B.append("UYARI: Gaz maliyeti verisi eklenmedi; eğri kayması 'maliyet' olarak yorumlanmamalı, 'aynı yükte fiyat seviyesi' olarak yazılmalı.")
if VAR["smf"]: B.append(f"SMF − PTF ({AY_ADI} {HEDEF_YIL}): {f(k.loc['SMF − PTF (USD/MWh)', c0])} USD/MWh.")
if VAR["sistem_yonu"]: B.append(f"Enerji açığı saat oranı ({AY_ADI} {HEDEF_YIL}): %{f(k.loc['Enerji açığı saatleri (%)', c0], 0)} ({AY_ADI} {HEDEF_YIL-1}: %{f(k.loc['Enerji açığı saatleri (%)', c1], 0)}).")
if ozet_k is not None: B.append(f"Kasım'ın ilk hafta içi günlerinde PTF (USD) {AY_ADI} hafta içi ortalamasına göre %{f(ozet_k.loc['PTF (USD/MWh)', 'Kasım/Ekim değişim (%)'], 0)} değişti.")
for b in B: print("-", b)

if "testler" in dir(): B.append("Neden testleri: " + "; ".join(f"{k}: kayma {v['Kayma (USD/MWh)']:+.1f} USD, tavan hariç {v['Kayma, tavan saatleri hariç']:+.1f}" for k, v in testler.items()))
ozet = {"kart": kart.astype(str).to_dict(), "testler": testler.astype(str).to_dict() if "testler" in dir() else {}, "ayristirma": ayr.drop(columns="hacim_aralik").astype(float).round(3).to_dict(orient="index"),
        "maliyet_verisi": MALIYET_VAR, "bulgular": B}
(OUT / "ozet_h02.json").write_text(json.dumps(ozet, ensure_ascii=False, indent=1), encoding="utf-8")
print("\nGrafikler ve özet:", OUT)